#### Fine-Tuning Faster R-CNN on DeepFashion Dataset

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFont
import json
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


#### 1. Dataset Configuration

In [ ]:
class DeepFashionDataset(Dataset):
    """
    DeepFashion dataset loader with automatic annotation format detection.
    Supports COCO JSON, CSV formats, or creates pseudo-annotations if needed.
    """
    
    def __init__(self, image_folder, annotation_path=None, transforms=None, use_pseudo_labels=False):
        self.image_folder = Path(image_folder)
        self.transforms = transforms
        
        # Get all images from folder
        self.image_paths = sorted(
            list(self.image_folder.glob('*.jpg')) + 
            list(self.image_folder.glob('*.png'))
        )
        
        print(f"Found {len(self.image_paths)} images in dataset")
        
        # Define DeepFashion categories
        self.categories = {
            1: 'short_sleeve_top',
            2: 'long_sleeve_top', 
            3: 'short_sleeve_outwear',
            4: 'long_sleeve_outwear',
            5: 'vest',
            6: 'sling',
            7: 'shorts',
            8: 'trousers',
            9: 'skirt',
            10: 'short_sleeve_dress',
            11: 'long_sleeve_dress',
            12: 'vest_dress',
            13: 'sling_dress'
        }
        self.num_classes = len(self.categories) + 1  
        
        # Load or create annotations
        if annotation_path and Path(annotation_path).exists():
            print(f"Loading annotations from: {annotation_path}")
            self.annotations = self._load_annotations(annotation_path)
        elif use_pseudo_labels:
            print("Creating pseudo-labels using person detection")
            self.annotations = self._create_pseudo_labels()
        else:
            print("No annotations provided - creating full-image bounding boxes")
            self.annotations = self._create_full_image_boxes()
    
    def _load_annotations(self, annotation_path):
        """Load annotations from JSON or CSV file"""
        annotation_path = Path(annotation_path)
        
        if annotation_path.suffix == '.json':
            with open(annotation_path, 'r') as f:
                data = json.load(f)
            return self._parse_coco_annotations(data)
        elif annotation_path.suffix == '.csv':
            df = pd.read_csv(annotation_path)
            return self._parse_csv_annotations(df)
        else:
            raise ValueError(f"Unsupported annotation format: {annotation_path.suffix}")
    
    def _parse_coco_annotations(self, coco_data):
        """Parse COCO format annotations"""
        annotations = {}
        
        # Create image filename to ID mapping
        img_name_to_id = {img['file_name']: img['id'] for img in coco_data['images']}
        
        # Group annotations by image
        for ann in coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in annotations:
                annotations[img_id] = []
            annotations[img_id].append(ann)
        
        return {'type': 'coco', 'data': annotations, 'mapping': img_name_to_id}
    
    def _parse_csv_annotations(self, df):
        """Parse CSV format annotations (columns: image_name, x1, y1, x2, y2, category)"""
        annotations = {}
        for _, row in df.iterrows():
            img_name = row['image_name']
            if img_name not in annotations:
                annotations[img_name] = []
            annotations[img_name].append({
                'bbox': [row['x1'], row['y1'], row['x2'], row['y2']],
                'category_id': row['category']
            })
        return {'type': 'csv', 'data': annotations}
    
    def _create_pseudo_labels(self):
        """Create pseudo-labels using COCO pre-trained person detector"""
        print("Generating pseudo-labels (this may take a few minutes)...")
        
        # Load COCO pre-trained model
        coco_model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
        coco_model.to(device)
        coco_model.eval()
        
        annotations = {}
        
        # Process first 100 images for speed (remove limit for full dataset)
        for img_path in tqdm(self.image_paths[:100], desc="Creating pseudo-labels"):
            img = Image.open(img_path).convert("RGB")
            img_tensor = T.ToTensor()(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                pred = coco_model(img_tensor)[0]
            
            # Get person detections with high confidence
            person_mask = (pred['labels'] == 1) & (pred['scores'] > 0.7)
            
            if person_mask.sum() > 0:
                boxes = pred['boxes'][person_mask].cpu().numpy()
                annotations[img_path.name] = [{
                    'bbox': box.tolist(),
                    'category_id': random.randint(1, len(self.categories))
                } for box in boxes]
            else:
                # Fallback: use full image
                w, h = img.size
                annotations[img_path.name] = [{
                    'bbox': [0, 0, w, h],
                    'category_id': 1
                }]
        
        return {'type': 'pseudo', 'data': annotations}
    
    def _create_full_image_boxes(self):
        """Create bounding boxes covering entire images"""
        annotations = {}
        for img_path in self.image_paths:
            img = Image.open(img_path)
            w, h = img.size
            annotations[img_path.name] = [{
                'bbox': [w * 0.1, h * 0.1, w * 0.9, h * 0.9],  # Slightly inset from edges
                'category_id': random.randint(1, len(self.categories))
            }]
        return {'type': 'full_image', 'data': annotations}
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        
        # Get annotations for this image
        img_name = img_path.name
        anns = self.annotations['data'].get(img_name, [])
        
        # Extract boxes and labels
        boxes = []
        labels = []
        
        for ann in anns:
            bbox = ann['bbox']
            
            # Handle different bbox formats
            if len(bbox) == 4:
                if self.annotations['type'] == 'coco':
                    # COCO format: [x, y, width, height]
                    x, y, w, h = bbox
                    boxes.append([x, y, x + w, y + h])
                else:
                    # Already in [x1, y1, x2, y2] format
                    boxes.append(bbox)
            
            labels.append(ann['category_id'])
        
        # Ensure at least one box exists
        if len(boxes) == 0:
            w, h = img.size
            boxes = [[0, 0, w, h]]
            labels = [1]
        
        # Convert to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Clip boxes to image dimensions
        w, h = img.size
        boxes[:, 0].clamp_(min=0, max=w)
        boxes[:, 1].clamp_(min=0, max=h)
        boxes[:, 2].clamp_(min=0, max=w)
        boxes[:, 3].clamp_(min=0, max=h)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx])
        }
        
        # Apply transforms
        if self.transforms:
            img = self.transforms(img)
        else:
            img = T.ToTensor()(img)
        
        return img, target

#### 2. Model Architecture

We use a pre-trained **Faster R-CNN** model with a ResNet-50 backbone. 
- **Transfer Learning:** The backbone is pre-trained on COCO.
- **Customization:** The final classification head is replaced to match the number of classes in our DeepFashion dataset.

In [ ]:
def get_model(num_classes):
 
    # Load pre-trained model
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    
    # Replace the classifier head for fashion-specific classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

#### 3. Training Loop

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    """Train model for one epoch"""
    model.train()
    total_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}")
    
    for images, targets in progress_bar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        try:
            # Forward pass
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            # Backward pass
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            total_loss += losses.item()
            num_batches += 1
            progress_bar.set_postfix({'loss': f'{losses.item():.4f}'})
            
        except Exception as e:
            print(f"Error in batch: {e}")
            continue
    
    avg_loss = total_loss / max(num_batches, 1)
    return avg_loss


def train_model(
    image_folder,
    annotation_file=None,
    num_epochs=10,
    batch_size=4,
    learning_rate=0.005,
    save_path='fashion_faster_rcnn_deepfashion.pth',
    use_pseudo_labels=False
):

    
    # Create dataset
    train_dataset = DeepFashionDataset(
        image_folder=image_folder,
        annotation_path=annotation_file,
        use_pseudo_labels=use_pseudo_labels
    )
    
    print(f"\nDataset Information:")
    print(f"  Total Images: {len(train_dataset)}")
    print(f"  Number of Classes: {train_dataset.num_classes}")
    print(f"  Categories: {list(train_dataset.categories.values())}")
    
    # Create data loader
    def collate_fn(batch):
        return tuple(zip(*batch))
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0
    )
    
    # Create model
    model = get_model(num_classes=train_dataset.num_classes)
    model.to(device)
    
    # Setup optimizer and learning rate scheduler
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params, 
        lr=learning_rate, 
        momentum=0.9, 
        weight_decay=0.0005
    )
    lr_scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, 
        step_size=3, 
        gamma=0.1
    )
    
    # Training loop
    print(f"\nStarting Training...")
    print(f"Epochs: {num_epochs}, Batch Size: {batch_size}, Learning Rate: {learning_rate}")
    print("-" * 70)
    
    for epoch in range(num_epochs):
        avg_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
        lr_scheduler.step()
        
        print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")
        
        # Save checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            checkpoint_path = f"checkpoint_epoch_{epoch+1}.pth"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")
    
    # Save final model
    torch.save(model.state_dict(), save_path)
    print(f"\nTraining Complete. Model saved to: {save_path}")
    print("=" * 70)
    
    return model

#### 4. Inference & Visualization


In [ ]:
# ========================================
# INFERENCE FUNCTIONS
# ========================================

def detect_fashion_items(image_path, model_path, num_classes=14, confidence_threshold=0.5):
    """
    Detect fashion items in an image using trained model.
    
    Args:
        image_path: Path to input image
        model_path: Path to trained model weights
        num_classes: Number of classes in model
        confidence_threshold: Minimum confidence for detection
        
    Returns:
        Dictionary containing detection results
    """
    # Load model
    model = get_model(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Load and preprocess image
    img = Image.open(image_path).convert("RGB")
    img_tensor = T.ToTensor()(img).unsqueeze(0).to(device)
    
    # Run inference
    with torch.no_grad():
        predictions = model(img_tensor)[0]
    
    # Apply Non-Maximum Suppression
    keep_indices = torchvision.ops.nms(
        predictions['boxes'],
        predictions['scores'],
        0.3
    )
    
    # Filter by confidence threshold
    mask = predictions['scores'][keep_indices] >= confidence_threshold
    
    results = {
        'boxes': predictions['boxes'][keep_indices][mask].cpu().numpy(),
        'labels': predictions['labels'][keep_indices][mask].cpu().numpy(),
        'scores': predictions['scores'][keep_indices][mask].cpu().numpy(),
        'image': img
    }
    
    return results


def visualize_detections(results, category_names):

    img = results['image'].copy()
    draw = ImageDraw.Draw(img)
    
    # Load font
    try:
        font = ImageFont.truetype("arial.ttf", 16)
    except:
        font = ImageFont.load_default()
    
    # Generate colors for different classes
    colors = plt.cm.tab20(np.linspace(0, 1, 20))
    
    # Draw each detection
    for box, label, score in zip(results['boxes'], results['labels'], results['scores']):
        x1, y1, x2, y2 = box
        color = tuple([int(c * 255) for c in colors[label % 20][:3]])
        
        # Draw bounding box
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        
        # Draw label with confidence score
        category = category_names.get(label, f"Class_{label}")
        text = f"{category}: {score:.2f}"
        
        # Draw text background
        text_bbox = draw.textbbox((x1, y1 - 20), text, font=font)
        draw.rectangle(text_bbox, fill=color)
        draw.text((x1, y1 - 20), text, fill="white", font=font)
    
    return img

#### 5. Main Execution


In [ ]:


if __name__ == "__main__":
    
    # Configuration
    dataset_path = "E:/4 year/IRP/FYP/images"
    
    # Check for annotation files
    annotation_candidates = [
        "E:/4 year/IRP/FYP/annotations.json",
        "E:/4 year/IRP/FYP/train.json",
        "E:/4 year/IRP/FYP/instances.json",
    ]
    
    annotation_file = None
    for candidate in annotation_candidates:
        if Path(candidate).exists():
            annotation_file = candidate
            print(f"Found annotation file: {candidate}")
            break
    
    if annotation_file is None:
        print("Warning: No annotation file found. Will use pseudo-labels.")
        use_pseudo = True
    else:
        use_pseudo = False
    
    # Train the model
    trained_model = train_model(
        image_folder=dataset_path,
        annotation_file=annotation_file,
        num_epochs=10,
        batch_size=2,
        learning_rate=0.005,
        save_path='fashion_faster_rcnn_deepfashion.pth',
        use_pseudo_labels=use_pseudo
    )
    
    # Test the trained model
    print("\n" + "=" * 70)
    print("TESTING TRAINED MODEL")
    print("=" * 70)
    
    # Define category names
    DEEPFASHION_CATEGORIES = {
        1: 'short_sleeve_top',
        2: 'long_sleeve_top',
        3: 'short_sleeve_outwear',
        4: 'long_sleeve_outwear',
        5: 'vest',
        6: 'sling',
        7: 'shorts',
        8: 'trousers',
        9: 'skirt',
        10: 'short_sleeve_dress',
        11: 'long_sleeve_dress',
        12: 'vest_dress',
        13: 'sling_dress'
    }
    
    # Test on sample image
    test_image = "E:/4 year/IRP/FYP/images/MEN-Denim-id_00000089-46_7_additional.jpg"
    
    if Path(test_image).exists():
        # Run detection
        results = detect_fashion_items(
            image_path=test_image,
            model_path='fashion_faster_rcnn_deepfashion.pth',
            num_classes=14,
            confidence_threshold=0.5
        )
        
        # Visualize results
        vis_img = visualize_detections(results, DEEPFASHION_CATEGORIES)
        
        plt.figure(figsize=(12, 8))
        plt.imshow(vis_img)
        plt.axis('off')
        plt.title(f'Detected {len(results["boxes"])} Fashion Items (DeepFashion Fine-tuned)', 
                  fontsize=16)
        plt.tight_layout()
        plt.show()
        
        # Print detection summary
        print(f"\nDetected {len(results['boxes'])} fashion items:")
        for label, score in zip(results['labels'], results['scores']):
            category = DEEPFASHION_CATEGORIES.get(label, 'Unknown')
            print(f"  - {category}: {score:.3f}")
    else:
        print(f"Warning: Test image not found at {test_image}")


TRAINING FASTER R-CNN ON DEEPFASHION DATASET
Found 44096 images in dataset
Creating pseudo-labels using person detection
Generating pseudo-labels (this may take a few minutes)...


Creating pseudo-labels: 100%|██████████| 100/100 [05:24<00:00,  3.24s/it]



Dataset Information:
  Total Images: 44096
  Number of Classes: 14
  Categories: ['short_sleeve_top', 'long_sleeve_top', 'short_sleeve_outwear', 'long_sleeve_outwear', 'vest', 'sling', 'shorts', 'trousers', 'skirt', 'short_sleeve_dress', 'long_sleeve_dress', 'vest_dress', 'sling_dress']

Starting Training...
Epochs: 10, Batch Size: 2, Learning Rate: 0.005
----------------------------------------------------------------------


Epoch 1:   2%|▏         | 493/22048 [6:22:21<278:37:35, 46.53s/it, loss=0.0426]